# 06 — Deduplicación y fusión

Este notebook aplica deduplicación y fusión únicamente dentro de la misma
`Fuente_origen` y para el mismo `Autor_norm`.

Reglas centrales:

- nunca fusionar registros de distintas `Fuente_origen`;
- nunca fusionar autores distintos;
- conservar la unidad publicación + autor;
- usar DOI/título/año/evidencia adicional para generar y clasificar candidatos;
- priorizar precisión: los casos ambiguos pasan a revisión manual;
- aplicar el archivo `casos_revision_deduplicacion_resueltos_final.csv`;
- conservar las 3 decisiones `NO_FUSION` como publicaciones/manifestaciones distintas;
- dentro de esas decisiones, eliminar repeticiones técnicas del mismo DOI sin cruzar DOI;
- usar el menor `indice` disponible como superviviente determinista y registrar los demás en auditoría;
- mantener exactamente 15 columnas canónicas en la salida;
- no completar faltantes generales ni mezclar información entre fuentes distintas.


In [ ]:
from pathlib import Path
import os
import pandas as pd
import re, unicodedata, itertools, hashlib, json
from collections import defaultdict

archivo_entrada = "../04_Limpieza/03_limpieza_bibliografica/autores_unam_limpios.csv"
carpeta_salida = "../04_Limpieza/04_deduplicacion"
os.makedirs(carpeta_salida, exist_ok=True)

archivo_candidatos = f"{carpeta_salida}/candidatos_deduplicacion.csv"
archivo_revision = f"{carpeta_salida}/casos_revision_deduplicacion.csv"
archivo_revision_resuelta = f"{carpeta_salida}/casos_revision_deduplicacion_resueltos_final.csv"
archivo_auditoria = f"{carpeta_salida}/auditoria_fusion.csv"
archivo_salida = f"{carpeta_salida}/autores_unam_deduplicados.csv"

CANON = ['Fuente_origen','indice','Titulo','Año','Autor_norm','Afiliacion1','Afiliacion2','ISBN','ISSN','Doi','URL','Area','SubArea','Keywords','Abstract']
AREAS = {'ISBD','CC','IA','TC','SIAV','RS'}

try:
    from rapidfuzz import fuzz
    def title_similarity(a,b):
        if not a or not b: return 0.0
        return max(fuzz.token_ratio(a,b), fuzz.WRatio(a,b))/100.0
except Exception:
    from difflib import SequenceMatcher
    def title_similarity(a,b):
        if not a or not b: return 0.0
        seq=SequenceMatcher(None,a,b).ratio()
        ta=' '.join(sorted(a.split())); tb=' '.join(sorted(b.split()))
        return max(seq,SequenceMatcher(None,ta,tb).ratio())

def clean(v): return '' if pd.isna(v) else str(v).strip()
def nfc(v): return unicodedata.normalize('NFC', clean(v))
def norm_title(v):
    x=nfc(v).casefold().replace('&amp;','&')
    x=re.sub(r'[^\w\s]',' ',x,flags=re.UNICODE).replace('_',' ')
    return re.sub(r'\s+',' ',x).strip()
def norm_doi(v):
    x=nfc(v).casefold().strip()
    x=re.sub(r'^https?://(?:dx\.)?doi\.org/','',x)
    x=re.sub(r'^doi\s*:\s*','',x)
    return x.strip()
def norm_url(v):
    x=nfc(v).strip()
    if not x:return ''
    return re.sub(r'#.*$','',x).rstrip('/').casefold()
def norm_abstract(v): return re.sub(r'\s+',' ',nfc(v)).strip().casefold()
def split_semicolon(v): return [p.strip() for p in re.split(r'\s*;\s*',clean(v)) if p.strip()]
def norm_id_token(v): return re.sub(r'[\s-]','',clean(v)).upper()
def id_set(v): return {norm_id_token(x) for x in split_semicolon(v) if norm_id_token(x)}
def compatible_year(a,b):
    a,b=clean(a),clean(b); return (not a) or (not b) or a==b

def support_features(a,b):
    out=[]
    if a['_url_n'] and a['_url_n']==b['_url_n']: out.append('URL_EXACTA')
    if id_set(a['ISBN']) & id_set(b['ISBN']): out.append('ISBN_COINCIDE')
    if id_set(a['ISSN']) & id_set(b['ISSN']): out.append('ISSN_COINCIDE')
    if a['_abs_n'] and a['_abs_n']==b['_abs_n']: out.append('ABSTRACT_EXACTO')
    return out

def valid_issn_token(t): return bool(re.fullmatch(r'\d{4}-\d{3}[\dXx]',clean(t)))
def valid_isbn_token(t):
    x=re.sub(r'[-\s]','',clean(t)); return bool(re.fullmatch(r'(?:\d{9}[\dXx]|\d{13})',x))
def valid_doi(v):
    x=norm_doi(v); return (not x) or bool(re.match(r'^10\.\d{4,9}/\S+$',x))

def choose_survivor(g):
    def k(row):
        idx=clean(row['indice'])
        try:return (0,int(float(idx)),int(row['_row_id']))
        except:return (1,idx.casefold(),int(row['_row_id']))
    return min((r for _,r in g.iterrows()),key=k)

def ordered_unique(values,normalizer=lambda x:clean(x)):
    seen=set();out=[]
    for v in values:
        v=clean(v)
        if not v:continue
        k=normalizer(v)
        if k not in seen:seen.add(k);out.append(v)
    return out

def merge_semicolon(values,case_insensitive=True):
    seen=set();out=[]
    for v in values:
        for token in split_semicolon(v):
            k=re.sub(r'\s+',' ',token).strip().casefold() if case_insensitive else token
            if k not in seen:seen.add(k);out.append(token)
    return '; '.join(out)

def group_conflicts(g):
    conflicts=[]
    if g['Fuente_origen'].nunique()!=1: conflicts.append('FUENTE_DISTINTA')
    if g['Autor_norm'].nunique()!=1: conflicts.append('AUTOR_DISTINTO')
    for col,norm,label in [('Año',clean,'ANIO_DISTINTO'),('Doi',norm_doi,'DOI_DISTINTO'),('Area',clean,'AREA_DISTINTA'),('URL',norm_url,'URL_DISTINTA'),('Abstract',norm_abstract,'ABSTRACT_DISTINTO')]:
        vals={norm(v) for v in g[col] if norm(v)}
        if len(vals)>1: conflicts.append(label)
    aff_sets=[]
    for _,r in g.iterrows():
        s={clean(v) for v in (r['Afiliacion1'],r['Afiliacion2']) if clean(v)}
        if s:aff_sets.append(s)
    union=set().union(*aff_sets) if aff_sets else set()
    if len(union)>2: conflicts.append('MAS_DE_DOS_AFILIACIONES')
    elif len(union)==2 and aff_sets and not any(s==union for s in aff_sets): conflicts.append('AFILIACIONES_DIFERENTES_SIN_ANCLA')
    return conflicts

def graph_connected(inds,edge_recs,edge_ok):
    if len(inds)<=1:return True
    adj={i:set() for i in inds}
    for r in edge_recs:
        if edge_ok(r):
            adj[r['i']].add(r['j']);adj[r['j']].add(r['i'])
    seen=set();stack=[inds[0]]
    while stack:
        x=stack.pop()
        if x in seen:continue
        seen.add(x);stack.extend(adj[x]-seen)
    return len(seen)==len(inds)

def fusion_group(g,overrides=None):
    base=choose_survivor(g); result={c:clean(base[c]) for c in CANON}
    complemented=[];combined=[]
    titles=ordered_unique(g['Titulo'],norm_title)
    if titles:
        chosen=max(titles,key=lambda x:(len(x),-titles.index(x)))
        if result['Titulo']!=chosen:
            combined.append('Titulo');result['Titulo']=chosen
    for col,norm in [('Año',clean),('Doi',norm_doi),('URL',norm_url),('Area',clean),('Abstract',norm_abstract)]:
        vals=ordered_unique(g[col],norm)
        if vals:
            chosen=max(vals,key=len) if col=='Abstract' else vals[0]
            if not result[col] and chosen: complemented.append(col)
            result[col]=chosen
    aff=[]
    for v in [base['Afiliacion1'],base['Afiliacion2']]:
        if clean(v) and clean(v) not in aff:aff.append(clean(v))
    for _,r in g.iterrows():
        for v in [r['Afiliacion1'],r['Afiliacion2']]:
            v=clean(v)
            if v and v not in aff:aff.append(v)
    old=(result['Afiliacion1'],result['Afiliacion2'])
    result['Afiliacion1']=aff[0] if aff else ''
    result['Afiliacion2']=aff[1] if len(aff)>1 else ''
    if old!=(result['Afiliacion1'],result['Afiliacion2']):
        if not old[1] and result['Afiliacion2']:complemented.append('Afiliacion2')
        else:combined.append('Afiliaciones')
    for col in ['ISBN','ISSN','Keywords']:
        merged=merge_semicolon(g[col])
        if merged!=result[col]:
            if not result[col] and merged:complemented.append(col)
            elif merged:combined.append(col)
            result[col]=merged
    if overrides:
        for col,val in overrides.items():
            if col in result and val is not None:
                result[col]=clean(val)
    result['SubArea']=''
    return result,{
        'indices_originales':' | '.join(map(str,g['indice'].tolist())),
        'row_ids_originales':' | '.join(map(str,g['_row_id'].astype(int).tolist())),
        'indice_sobreviviente':clean(base['indice']),
        'row_id_sobreviviente':str(int(base['_row_id'])),
        'campos_complementados':'; '.join(sorted(set(complemented))),
        'campos_combinados':'; '.join(sorted(set(combined))),
    }

def gid(prefix,g):
    sig='|'.join(map(str,sorted(g['_row_id'].astype(int))))
    return prefix+'_'+hashlib.sha1(sig.encode()).hexdigest()[:10].upper()

# ---------- load ----------
df0=pd.read_csv(archivo_entrada,dtype=str,keep_default_na=False)
missing=[c for c in CANON if c not in df0.columns]
if missing:raise ValueError(missing)
extra=[c for c in df0.columns if c not in CANON]
nonempty=[c for c in extra if (df0[c].astype(str).str.strip()!='').any()]
if nonempty:raise ValueError(f'Columnas extra con información: {nonempty}')
df=df0[CANON].copy()
for c in CANON:df[c]=df[c].map(clean)
df.insert(0,'_row_id',range(1,len(df)+1))

# invalid ISBN inherited (do not repair in this phase)
invalid_isbn_input=[]
for idx,v in df['ISBN'].items():
    for t in split_semicolon(v):
        if not valid_isbn_token(t):invalid_isbn_input.append((idx,t))

# ---------- level 1: exact duplicates first ----------
exact_groups=[]; drop_exact=set(); audit=[]; candidate_rows=[]
for _,g in df.groupby(CANON,dropna=False,sort=False):
    if len(g)<=1:continue
    base=choose_survivor(g)
    survivors_idx=base.name
    others=[i for i in g.index if i!=survivors_idx]
    drop_exact.update(others)
    egid=gid('E',g)
    exact_groups.append((egid,list(g.index)))
    candidate_rows.append({
        'Grupo_ID':egid,'Etapa':'EXACTO','Decision':'AUTO_FUSION','Fuente_origen':g['Fuente_origen'].iloc[0],
        'indices':' | '.join(g['indice'].tolist()),'row_ids':' | '.join(g['_row_id'].astype(str).tolist()),'Autor_norm':g['Autor_norm'].iloc[0],
        'n_filas':len(g),'Titulos_distintos':g['Titulo'].iloc[0],'Años_distintos':g['Año'].iloc[0],'DOIs_distintos':g['Doi'].iloc[0],
        'ISBNs_distintos':g['ISBN'].iloc[0],'ISSNs_distintos':g['ISSN'].iloc[0],'URLs_distintas':g['URL'].iloc[0],
        'Areas_distintas':g['Area'].iloc[0],'Afiliaciones_distintas':' || '.join(ordered_unique([v for _,r in g.iterrows() for v in (r['Afiliacion1'],r['Afiliacion2'])])),
        'Abstracts_no_vacios_distintos':1 if norm_abstract(g['Abstract'].iloc[0]) else 0,'Similitud_titulo_min':1.0,'Similitud_titulo_max':1.0,
        'Tipos_candidato':'EXACTO_15','Reglas_candidatura':'15_COLUMNAS_IDENTICAS','Evidencia_adicional':'15_COLUMNAS_IDENTICAS','Conflictos_detectados':'',
        'Motivo':'Duplicado técnico: 15 columnas idénticas.'})
    audit.append({'Grupo_ID':egid,'Fuente_origen':g['Fuente_origen'].iloc[0],'indices_originales':' | '.join(g['indice'].tolist()),
        'row_ids_originales':' | '.join(g['_row_id'].astype(str).tolist()),'indice_sobreviviente':clean(base['indice']),'row_id_sobreviviente':str(int(base['_row_id'])),
        'Autor_norm':g['Autor_norm'].iloc[0],'Doi':g['Doi'].iloc[0],'Titulo':g['Titulo'].iloc[0],'Regla_utilizada':'15 columnas idénticas',
        'Campos_complementados':'','Campos_combinados':'','Conflictos_detectados':'','Decision':'AUTO_FUSION_EXACTA','Filas_entrada_grupo':len(g),'Filas_absorbidas':len(g)-1})

work=df.drop(index=list(drop_exact)).copy()
for c,fun in [('_title_n',norm_title),('_doi_n',norm_doi),('_url_n',norm_url),('_abs_n',norm_abstract)]:
    source={'_title_n':'Titulo','_doi_n':'Doi','_url_n':'URL','_abs_n':'Abstract'}[c]
    work[c]=work[source].map(fun)

# ---------- candidate pairs ----------
pairs={}
def add_pair(i,j,kind,rule,sim=None,support=None):
    if i==j:return
    if i>j:i,j=j,i
    r=pairs.setdefault((i,j),{'i':i,'j':j,'kinds':set(),'rules':set(),'support':set(),'sim':None})
    r['kinds'].add(kind);r['rules'].add(rule)
    if support:r['support'].update(support)
    if sim is not None:r['sim']=max(r['sim'] or 0.0,float(sim))

for _,g in work[work['_doi_n']!=''].groupby(['Fuente_origen','Autor_norm','_doi_n'],sort=False):
    if len(g)>1:
        for i,j in itertools.combinations(g.index,2):
            a,b=work.loc[i],work.loc[j];sim=title_similarity(a['_title_n'],b['_title_n'])
            kind='DOI_MISMO_TITULO_EXACTO' if a['_title_n']==b['_title_n'] else 'DOI_MISMO_TITULO_VARIANTE'
            add_pair(i,j,kind,'MISMA_FUENTE_AUTOR_DOI',sim,support_features(a,b))

for _,g in work.groupby(['Fuente_origen','Autor_norm','_title_n'],sort=False):
    if len(g)>1:
        for i,j in itertools.combinations(g.index,2):
            a,b=work.loc[i],work.loc[j]
            if not compatible_year(a['Año'],b['Año']):continue
            da,db=a['_doi_n'],b['_doi_n']
            if da and db and da==db:continue
            sup=support_features(a,b)
            if not da and not db:add_pair(i,j,'SIN_DOI_TITULO_EXACTO','MISMA_FUENTE_AUTOR_TITULO_ANIO_COMPATIBLE',1.0,sup)
            elif bool(da)^bool(db):add_pair(i,j,'DOI_VS_VACIO_TITULO_EXACTO','DOI_PRESENTE_VS_VACIO_MISMO_TITULO',1.0,sup)
            else:add_pair(i,j,'DOI_DISTINTO_TITULO_EXACTO','DOI_CONFLICTIVO_MISMO_TITULO',1.0,sup)

no_doi=work[work['_doi_n']=='']
for _,g in no_doi.groupby(['Fuente_origen','Autor_norm','Año'],sort=False):
    if len(g)<2:continue
    for i,j in itertools.combinations(g.index,2):
        a,b=work.loc[i],work.loc[j]
        if a['_title_n']==b['_title_n'] or not a['_title_n'] or not b['_title_n']:continue
        sim=title_similarity(a['_title_n'],b['_title_n'])
        if sim>=0.93:add_pair(i,j,'SIN_DOI_TITULO_APROX','FUZZY_DENTRO_BLOQUE_FUENTE_AUTOR_ANIO',sim,support_features(a,b))

# ---------- groups and classification ----------
parent={i:i for i in work.index}
def find(x):
    while parent[x]!=x:parent[x]=parent[parent[x]];x=parent[x]
    return x
def union(a,b):
    ra,rb=find(a),find(b)
    if ra!=rb:parent[rb]=ra
for r in pairs.values():union(r['i'],r['j'])
components=defaultdict(list)
for i in sorted({x for p in pairs for x in p}):components[find(i)].append(i)
components=sorted(components.values(),key=lambda xs:min(int(work.loc[i,'_row_id']) for i in xs))

auto_groups=[];review_groups=[]
for inds in components:
    g=work.loc[inds].copy();edges=[r for r in pairs.values() if r['i'] in inds and r['j'] in inds]
    kinds=sorted(set().union(*(r['kinds'] for r in edges)));rules=sorted(set().union(*(r['rules'] for r in edges)));supports=sorted(set().union(*(r['support'] for r in edges)))
    sims=[r['sim'] for r in edges if r['sim'] is not None]; conflicts=group_conflicts(g)
    dois={norm_doi(v) for v in g['Doi'] if norm_doi(v)};titles={norm_title(v) for v in g['Titulo'] if norm_title(v)};years={clean(v) for v in g['Año'] if clean(v)}
    has_fuzzy='SIN_DOI_TITULO_APROX' in kinds; has_doi_conflict=len(dois)>1 or 'DOI_DISTINTO_TITULO_EXACTO' in kinds; title_min=min(sims) if sims else 1.0
    if has_doi_conflict:
        decision='REVISION';motivo='Existen DOI distintos no vacíos dentro del grupo candidato.'
    elif dois:
        def safe(r):
            if any(k.startswith('DOI_MISMO') for k in r['kinds']):return True
            if 'DOI_VS_VACIO_TITULO_EXACTO' in r['kinds'] and bool(r['support']):return True
            return False
        if title_min<0.94:decision='REVISION';motivo='Mismo DOI y autor, pero el título presenta diferencia sustancial.'
        elif conflicts:decision='REVISION';motivo='Mismo DOI, pero existe al menos una contradicción no resoluble automáticamente.'
        elif not graph_connected(inds,edges,safe):decision='REVISION';motivo='El grupo contiene una fila con DOI vacío sin evidencia independiente suficiente para adherirla automáticamente.'
        else:decision='AUTO_FUSION';motivo='Misma fuente + mismo autor + un único DOI + título equivalente y conectividad completa por evidencia segura.'
    elif has_fuzzy:
        decision='REVISION';motivo='La similitud aproximada solo se usa para descubrir candidatos; requiere confirmación.'
    elif len(titles)==1 and len(years)<=1:
        def safe(r):return 'SIN_DOI_TITULO_EXACTO' in r['kinds'] and bool(r['support'])
        if conflicts:decision='REVISION';motivo='Título exacto y autor/fuente coincidentes, pero existe contradicción.'
        elif graph_connected(inds,edges,safe):decision='AUTO_FUSION';motivo='Sin DOI: título exacto + autor + fuente + año compatible y conectividad completa por evidencia bibliográfica adicional.'
        else:decision='REVISION';motivo='Sin DOI: coincidencia exacta, pero al menos una fila carece de evidencia adicional suficiente.'
    else:
        decision='REVISION';motivo='Evidencia insuficiente o patrón no cubierto por reglas automáticas conservadoras.'
    G=gid('G',g)
    row={'Grupo_ID':G,'Etapa':'MATCHING','Decision':decision,'Fuente_origen':g['Fuente_origen'].iloc[0] if g['Fuente_origen'].nunique()==1 else ' | '.join(ordered_unique(g['Fuente_origen'])),
         'indices':' | '.join(g['indice'].tolist()),'row_ids':' | '.join(g['_row_id'].astype(str).tolist()),'Autor_norm':g['Autor_norm'].iloc[0] if g['Autor_norm'].nunique()==1 else ' | '.join(ordered_unique(g['Autor_norm'])),
         'n_filas':len(g),'Titulos_distintos':' || '.join(ordered_unique(g['Titulo'],norm_title)),'Años_distintos':' | '.join(ordered_unique(g['Año'])),'DOIs_distintos':' | '.join(ordered_unique(g['Doi'],norm_doi)),
         'ISBNs_distintos':' | '.join(ordered_unique(g['ISBN'])),'ISSNs_distintos':' | '.join(ordered_unique(g['ISSN'])),'URLs_distintas':' || '.join(ordered_unique(g['URL'],norm_url)),
         'Areas_distintas':' | '.join(ordered_unique(g['Area'])),'Afiliaciones_distintas':' || '.join(ordered_unique([v for _,r in g.iterrows() for v in (r['Afiliacion1'],r['Afiliacion2'])])),
         'Abstracts_no_vacios_distintos':len({norm_abstract(v) for v in g['Abstract'] if norm_abstract(v)}),'Similitud_titulo_min':round(title_min,6),'Similitud_titulo_max':round(max(sims) if sims else 1.0,6),
         'Tipos_candidato':'; '.join(kinds),'Reglas_candidatura':'; '.join(rules),'Evidencia_adicional':'; '.join(supports),'Conflictos_detectados':'; '.join(conflicts),'Motivo':motivo}
    candidate_rows.append(row);(auto_groups if decision=='AUTO_FUSION' else review_groups).append((G,inds,row))

candidates=pd.DataFrame(candidate_rows)
candidates.to_csv(archivo_candidatos,index=False,encoding='utf-8-sig')

manual_cols=['Decision_manual','Titulo_manual','Año_manual','Doi_manual','URL_manual','Area_manual','Abstract_manual','Afiliacion1_manual','Afiliacion2_manual','Comentario']
review_base=candidates[candidates['Decision']=='REVISION'].copy()
for c in manual_cols:
    review_base[c]='REVISAR' if c=='Decision_manual' else ''

review_cols=['Grupo_ID','Fuente_origen','indices','row_ids','Autor_norm','n_filas','Titulos_distintos','Años_distintos','DOIs_distintos','ISBNs_distintos','ISSNs_distintos','URLs_distintas','Areas_distintas','Afiliaciones_distintas','Abstracts_no_vacios_distintos','Similitud_titulo_min','Tipos_candidato','Evidencia_adicional','Conflictos_detectados','Motivo']+manual_cols
reviews=review_base[review_cols]
reviews.to_csv(archivo_revision,index=False,encoding='utf-8-sig')



# ---------- cargar y validar revisión manual resuelta ----------
if not os.path.exists(archivo_revision_resuelta):
    raise FileNotFoundError(
        f'Falta el archivo de revisión manual resuelta: {archivo_revision_resuelta}'
    )

resolved=pd.read_csv(archivo_revision_resuelta,dtype=str,keep_default_na=False)

required_resolved = [
    'Grupo_ID','Fuente_origen','indices','row_ids','Autor_norm','n_filas',
    'Titulos_distintos','Años_distintos','DOIs_distintos','ISBNs_distintos',
    'ISSNs_distintos','URLs_distintas','Areas_distintas','Afiliaciones_distintas',
    'Abstracts_no_vacios_distintos','Similitud_titulo_min','Tipos_candidato',
    'Evidencia_adicional','Conflictos_detectados','Motivo','Decision_manual',
    'Titulo_manual','Año_manual','Doi_manual','ISBN_manual','URL_manual',
    'Area_manual','Abstract_manual','Afiliacion1_manual','Afiliacion2_manual',
    'Comentario'
]
missing_resolved=[c for c in required_resolved if c not in resolved.columns]
if missing_resolved:
    raise ValueError(f'Faltan columnas en revisión resuelta: {missing_resolved}')

if resolved['Grupo_ID'].duplicated().any():
    raise ValueError('La revisión resuelta contiene Grupo_ID duplicado.')

expected_review_ids=set(reviews['Grupo_ID'])
resolved_ids=set(resolved['Grupo_ID'])
if expected_review_ids != resolved_ids:
    raise ValueError(
        f'La revisión resuelta no corresponde exactamente a los grupos actuales. '
        f'Faltan={sorted(expected_review_ids-resolved_ids)[:10]}, '
        f'Sobran={sorted(resolved_ids-expected_review_ids)[:10]}'
    )

allowed_manual={'FUSIONAR','NO_FUSION'}
bad_decisions=set(resolved['Decision_manual'])-allowed_manual
if bad_decisions:
    raise ValueError(f'Decisiones manuales no permitidas: {sorted(bad_decisions)}')

if (resolved['Decision_manual'].str.strip()=='').any():
    raise ValueError('Existen decisiones manuales vacías.')

resolved_idx=resolved.set_index('Grupo_ID')

# ---------- plan final de fusiones ----------
# Exactos y AUTO_FUSION ya detectados por reglas conservadoras.
audit=[]
final_groups=[]   # tuples (orden, Grupo_ID, DataFrame, decision, regla, overrides)

# 1) Exactos técnicos
for egid, idxs in exact_groups:
    g=df.loc[idxs].copy()
    final_groups.append((
        min(g.index), egid, g, 'AUTO_FUSION_EXACTA',
        '15 columnas idénticas', {}
    ))

# 2) Matching automático
for G,inds,meta in auto_groups:
    g=work.loc[inds].copy()
    conf=group_conflicts(g)
    if conf:
        raise AssertionError((G,conf))
    final_groups.append((
        min(g.index), G, g, 'AUTO_FUSION',
        meta['Motivo'], {}
    ))

# 3) Matching manual resuelto
manual_no_fusion_ids=set()

def manual_overrides(row):
    mapping={
        'Titulo_manual':'Titulo',
        'Año_manual':'Año',
        'Doi_manual':'Doi',
        'ISBN_manual':'ISBN',
        'URL_manual':'URL',
        'Area_manual':'Area',
        'Abstract_manual':'Abstract',
        'Afiliacion1_manual':'Afiliacion1',
        'Afiliacion2_manual':'Afiliacion2',
    }
    out={}
    # Afiliaciones: si la revisión manual especifica al menos una de las dos,
    # se interpreta como decisión completa del par de afiliaciones.
    # Una celda vacía o __VACIO__ significa que esa posición debe quedar vacía.
    aff1_raw=clean(row['Afiliacion1_manual'])
    aff2_raw=clean(row['Afiliacion2_manual'])
    aff_manual=bool(aff1_raw or aff2_raw)

    for src,dst in mapping.items():
        if src in {'Afiliacion1_manual','Afiliacion2_manual'}:
            continue
        v=clean(row[src])
        if not v:
            continue
        out[dst] = '' if v=='__VACIO__' else v

    if aff_manual:
        out['Afiliacion1']='' if aff1_raw in {'','__VACIO__'} else aff1_raw
        out['Afiliacion2']='' if aff2_raw in {'','__VACIO__'} else aff2_raw

    return out

for G,inds,meta in review_groups:
    decision=resolved_idx.loc[G]
    g=work.loc[inds].copy()

    if decision['Decision_manual']=='FUSIONAR':
        final_groups.append((
            min(g.index), G, g, 'MANUAL_FUSION',
            clean(decision['Comentario']) or 'Fusión aprobada manualmente.',
            manual_overrides(decision)
        ))
    else:
        manual_no_fusion_ids.update(inds)

        # La decisión NO_FUSION aplica al componente conflictivo completo:
        # conserva manifestaciones bibliográficas distintas.
        # Sin embargo, dentro de cada DOI confirmado pueden existir repeticiones
        # técnicas del mismo artículo-autor-fuente. Esas repeticiones se fusionan
        # por separado, sin cruzar DOI.
        if g['_doi_n'].eq('').any():
            raise ValueError(
                f'Grupo {G} marcado NO_FUSION contiene DOI vacío. '
                'Requiere una regla manual más específica antes de continuar.'
            )

        for doi_n, sg in g.groupby('_doi_n',sort=False):
            if sg['Fuente_origen'].nunique()!=1 or sg['Autor_norm'].nunique()!=1:
                raise ValueError(f'Subgrupo inconsistente dentro de {G}.')
            if len({norm_title(v) for v in sg['Titulo'] if norm_title(v)})!=1:
                raise ValueError(
                    f'Subgrupo DOI {doi_n} dentro de {G} contiene títulos distintos.'
                )
            if len(sg)>1:
                SG=G+'_DOI_'+hashlib.sha1(doi_n.encode()).hexdigest()[:8].upper()
                final_groups.append((
                    min(sg.index), SG, sg, 'FUSION_DENTRO_NO_FUSION',
                    f'El componente {G} se conserva separado por DOI según revisión manual; '
                    f'dentro del DOI {doi_n} las filas son repeticiones del mismo registro.',
                    {}
                ))

# ---------- comprobar que ningún row se absorba por dos planes incompatibles ----------
membership=defaultdict(list)
for _,G,g,decision,regla,over in final_groups:
    for i in g.index:
        membership[i].append(G)

# Solapamiento permitido: superviviente de un grupo EXACTO puede participar después
# en un grupo matching. Las copias exactas absorbidas no deben aparecer en matching.
exact_member_to_survivor={}
for egid,idxs in exact_groups:
    g=df.loc[idxs]
    survivor=choose_survivor(g).name
    for i in idxs:
        exact_member_to_survivor[i]=survivor

for i,groups in membership.items():
    if len(groups)>2:
        raise ValueError(f'Fila {i} participa en demasiados grupos: {groups}')
    if len(groups)==2:
        exact_groups_here=[g for g in groups if g.startswith('E_')]
        if len(exact_groups_here)!=1 or exact_member_to_survivor.get(i)!=i:
            raise ValueError(f'Solapamiento de fusión no permitido para fila {i}: {groups}')

# ---------- ejecutar secuencialmente: exactos -> matching auto/manual ----------
# Primero eliminar copias exactas y conservar sus supervivientes.
active=df.copy()
audit_rows=[]

for egid,idxs in exact_groups:
    g=active.loc[[i for i in idxs if i in active.index]].copy()
    if len(g)<=1:
        continue
    base=choose_survivor(g)
    survivor=base.name
    fused,m=fusion_group(g)
    # exactos: la fila resultante debe ser idéntica al superviviente
    active.loc[survivor,CANON]=[fused[c] for c in CANON]
    drop=[i for i in g.index if i!=survivor]
    active=active.drop(index=drop)

    audit_rows.append({
        'Grupo_ID':egid,
        'Fuente_origen':fused['Fuente_origen'],
        'indices_originales':m['indices_originales'],
        'row_ids_originales':m['row_ids_originales'],
        'indice_sobreviviente':m['indice_sobreviviente'],
        'row_id_sobreviviente':m['row_id_sobreviviente'],
        'Autor_norm':fused['Autor_norm'],
        'Doi':fused['Doi'],
        'Titulo':fused['Titulo'],
        'Regla_utilizada':'15 columnas idénticas',
        'Campos_complementados':'',
        'Campos_combinados':'',
        'Conflictos_detectados':'',
        'Decision':'AUTO_FUSION_EXACTA',
        'Filas_entrada_grupo':len(g),
        'Filas_absorbidas':len(g)-1
    })

# Resolver los grupos no exactos usando únicamente filas supervivientes activas.
# Los índices de work ya corresponden a esas filas.
matching_plans=[]
for G,inds,meta in auto_groups:
    matching_plans.append((G,inds,'AUTO_FUSION',meta['Motivo'],{}))

for G,inds,meta in review_groups:
    decision=resolved_idx.loc[G]
    if decision['Decision_manual']=='FUSIONAR':
        matching_plans.append((
            G,inds,'MANUAL_FUSION',
            clean(decision['Comentario']) or 'Fusión aprobada manualmente.',
            manual_overrides(decision)
        ))
    else:
        g=work.loc[inds].copy()
        for doi_n, sg in g.groupby('_doi_n',sort=False):
            if len(sg)>1:
                SG=G+'_DOI_'+hashlib.sha1(doi_n.encode()).hexdigest()[:8].upper()
                matching_plans.append((
                    SG,list(sg.index),'FUSION_DENTRO_NO_FUSION',
                    f'NO_FUSION manual del componente {G}; '
                    f'fusión limitada al mismo DOI {doi_n}.',
                    {}
                ))

used_nonexact=set()

for G,inds,decision_label,regla,overrides in matching_plans:
    present=[i for i in inds if i in active.index]
    if len(present)<2:
        continue
    if used_nonexact & set(present):
        raise ValueError(
            f'Un grupo no exacto solapa otro grupo ya fusionado: {G}'
        )

    g=active.loc[present].copy()
    if g['Fuente_origen'].nunique()!=1:
        raise ValueError(f'{G}: mezcla Fuente_origen.')
    if g['Autor_norm'].nunique()!=1:
        raise ValueError(f'{G}: mezcla Autor_norm.')

    if decision_label=='AUTO_FUSION':
        conf=group_conflicts(g)
        if conf:
            raise ValueError(f'{G}: AUTO_FUSION con conflictos: {conf}')

    fused,m=fusion_group(g,overrides=overrides)

    if fused['Fuente_origen']!=g['Fuente_origen'].iloc[0]:
        raise ValueError(f'{G}: cambió Fuente_origen.')
    if fused['Autor_norm']!=g['Autor_norm'].iloc[0]:
        raise ValueError(f'{G}: cambió Autor_norm.')

    # Nunca permitir más de dos afiliaciones ni afiliaciones repetidas.
    aff=[x for x in [fused['Afiliacion1'],fused['Afiliacion2']] if clean(x)]
    if len(set(aff))!=len(aff) or len(aff)>2:
        raise ValueError(f'{G}: afiliaciones finales inválidas: {aff}')

    base=choose_survivor(g)
    survivor=base.name
    active.loc[survivor,CANON]=[fused[c] for c in CANON]
    drop=[i for i in g.index if i!=survivor]
    active=active.drop(index=drop)
    used_nonexact.update(present)

    audit_rows.append({
        'Grupo_ID':G,
        'Fuente_origen':fused['Fuente_origen'],
        'indices_originales':m['indices_originales'],
        'row_ids_originales':m['row_ids_originales'],
        'indice_sobreviviente':m['indice_sobreviviente'],
        'row_id_sobreviviente':m['row_id_sobreviviente'],
        'Autor_norm':fused['Autor_norm'],
        'Doi':fused['Doi'],
        'Titulo':fused['Titulo'],
        'Regla_utilizada':regla,
        'Campos_complementados':m['campos_complementados'],
        'Campos_combinados':m['campos_combinados'],
        'Conflictos_detectados':'',
        'Decision':decision_label,
        'Filas_entrada_grupo':len(g),
        'Filas_absorbidas':len(g)-1
    })

# ---------- salida final ----------
out=active[CANON].copy().sort_index().reset_index(drop=True)
audit_df=pd.DataFrame(audit_rows)

out.to_csv(archivo_salida,index=False,encoding='utf-8-sig')
audit_df.to_csv(archivo_auditoria,index=False,encoding='utf-8-sig')

# ---------- validaciones finales fuertes ----------
assert list(out.columns)==CANON
assert len(out.columns)==15
assert set(out['Fuente_origen']).issubset(set(df['Fuente_origen']))
assert set(out['Autor_norm']).issubset(set(df['Autor_norm']))
assert (out['SubArea'].str.strip()=='').all()
assert set(out['Area'])<=AREAS
assert len(out)<=len(df)

absorbed=int(audit_df['Filas_absorbidas'].astype(int).sum())
assert len(df)-len(out)==absorbed

# Ninguna auditoría mezcla fuente o autor.
for _,r in audit_df.iterrows():
    ids=[int(x.strip()) for x in clean(r['row_ids_originales']).split('|') if x.strip()]
    original=df[df['_row_id'].isin(ids)]
    assert original['Fuente_origen'].nunique()==1
    assert original['Autor_norm'].nunique()==1

# Cada índice sobreviviente pertenece realmente a una fila original
# de la misma fuente y autor.
keys=set(df[['Fuente_origen','Autor_norm','indice']].itertuples(index=False,name=None))
assert all(
    tuple(x) in keys
    for x in out[['Fuente_origen','Autor_norm','indice']].itertuples(index=False,name=None)
)

invalid_doi=out.loc[~out['Doi'].map(valid_doi)]
invalid_issn=[];invalid_isbn=[]
for idx,v in out['ISSN'].items():
    for t in split_semicolon(v):
        if not valid_issn_token(t):invalid_issn.append((idx,t))
for idx,v in out['ISBN'].items():
    for t in split_semicolon(v):
        if not valid_isbn_token(t):invalid_isbn.append((idx,t))

assert not len(invalid_doi)
assert not invalid_issn

# La fase de deduplicación no puede introducir ISBN inválidos nuevos.
assert {t for _,t in invalid_isbn}.issubset({t for _,t in invalid_isbn_input})

# No queda ninguna revisión sin resolver.
assert set(resolved['Decision_manual']) <= {'FUSIONAR','NO_FUSION'}
assert len(resolved)==len(reviews)

# Los tres componentes NO_FUSION conservan más de un DOI final.
for G in resolved.loc[resolved['Decision_manual']=='NO_FUSION','Grupo_ID']:
    ids=[int(x.strip()) for x in resolved_idx.loc[G,'row_ids'].split('|') if x.strip()]
    original=work.loc[[i for i in work.index if int(work.loc[i,'_row_id']) in ids]]
    dois={norm_doi(x) for x in original['Doi'] if norm_doi(x)}
    if len(dois)<2:
        raise ValueError(f'{G}: NO_FUSION ya no representa múltiples DOI.')

summary={
    'filas_entrada':len(df),
    'duplicados_exactos_grupos':len(exact_groups),
    'grupos_auto_fusion_matching':len(auto_groups),
    'grupos_revision_resueltos':len(resolved),
    'grupos_manual_fusion':int((resolved['Decision_manual']=='FUSIONAR').sum()),
    'grupos_manual_no_fusion':int((resolved['Decision_manual']=='NO_FUSION').sum()),
    'filas_absorbidas_totales':absorbed,
    'filas_salida_final':len(out),
    'doi_invalidos_salida':len(invalid_doi),
    'issn_tokens_invalidos_salida':len(invalid_issn),
    'isbn_tokens_invalidos_salida':len(invalid_isbn),
    'estado':'FINAL'
}
print(json.dumps(summary,ensure_ascii=False,indent=2))
print('\nDecisiones manuales:')
print(resolved['Decision_manual'].value_counts().to_string())
print('\nArchivos generados:')
print('-',archivo_candidatos)
print('-',archivo_revision)
print('-',archivo_revision_resuelta)
print('-',archivo_auditoria)
print('-',archivo_salida)
